## reading bronze sales files

In [0]:
create or refresh streaming table sales
as
select *, current_timestamp() as ingestion_time from stream(read_files('s3://databricks-jayati-demo1234/dlt/', format =>'csv', header => true));

#Cleaning null values from sales and creating sales silver table

In [0]:
create or refresh streaming table sales_silver
(
    constraint valid_sales expect(order_id is not null) on violation drop row
)
as 
select distinct *  from stream(sales);

## Bronze Customer table

In [0]:
create or refresh streaming table customers
as
select *, current_timestamp() as ingestion_time from stream(read_files('s3://databricks-jayati-demo1234/dlt/', format =>'csv', header => true));

#Cleaning null values from  bronze customers and creating validated customer silver table

In [0]:
create OR refresh streaming table validated_customers_silver
(
    constraint valid_customer expect(customer_id is not null) on violation drop row
)
comment " validated silver customer table";

##scd 2 on  validated_customers_silver table

In [0]:
CREATE FLOW customers_cdc AS AUTO CDC INTO validated_customers_silver
FROM STREAM(customers)
KEYS (customer_id)
APPLY AS DELETE WHEN Operation = 'DELETE'
SEQUENCE BY sequenceNum
COLUMNS * EXCEPT (ingestion_time, sequenceNum, Operation, _rescued_data)
STORED AS SCD TYPE 2;

# bronze product table

In [0]:
create or refresh streaming table products
as
select *, current_timestamp() as ingestion_time from stream(read_files('s3://databricks-jayati-demo1234/dlt/', format =>'csv', header => true));

#Cleaning null values from  bronze customers and creating validated product silver table

In [0]:
create OR refresh streaming table validated_products_silver
(
    constraint valid_product expect(product_id is not null) on violation drop row

)
comment " validated silver product table"


##SCD1 on product table

In [0]:
CREATE FLOW products_cdc AS AUTO CDC INTO validated_products_silver
FROM STREAM(LIVE.products)
WITH (skipChangeCommits = "true")
KEYS (product_id)
APPLY AS DELETE WHEN operation = 'DELETE'
SEQUENCE BY seqNum
COLUMNS * EXCEPT (ingestion_time, seqNum, operation, _rescued_data)
STORED AS SCD TYPE 1;